# Files for IGV

One objective: turn the pipeline's outputs into sorted, compressed, indexed files that IGV can load: the predicted ORFs, the filtered transcripts, and (when you have them) the alignments.

In [1]:
import sys, glob, os
sys.path.insert(0, "src")
from igv_prepare import index_annotation, psl_to_bed12, prepare_bam, check_index

results_dir = "../LRP2_test_results"   # change to the results folder of the run
bam_dir = None                          # folder with the alignment BAM files, or None if you have none
out_dir = "igv"

**Explanation.** IGV reads large files fast only when they are sorted by position, compressed with bgzip and have an index file next to them (`.tbi` for annotation, `.bai` for BAM). The functions in `src/igv_prepare.py` do these steps. `bam_dir` is the one setting that depends on where your alignments are.

In [2]:
gtf = glob.glob(f"{results_dir}/*collapsed_high_confidence_ORF.gtf")[0]
bed = glob.glob(f"{results_dir}/*collapsed_high_confidence_ORF.bed")[0]
gtf_gz = index_annotation(gtf, out_dir, "gff")
bed_gz = index_annotation(bed, out_dir, "bed")
gtf_gz, bed_gz

('igv/SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.gtf.gz',
 'igv/SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.bed.gz')

**Explanation.** The predicted ORFs come as a GTF (genes and exons) and a BED (one line per ORF). Both are sorted, compressed and indexed into the `igv/` folder.

In [3]:
psl = glob.glob(f"{results_dir}/*transcriptome.filtered.psl")[0]
os.makedirs(out_dir, exist_ok=True)
bed12 = f"{out_dir}/" + os.path.basename(psl).replace(".psl", ".bed")
n = psl_to_bed12(psl, bed12)
transcripts_gz = index_annotation(bed12, out_dir, "bed")
print(n, "filtered transcripts ->", transcripts_gz)

305 filtered transcripts -> igv/SERVES_LRP2_test.transcriptome.filtered.bed.gz


**Explanation.** The filtered transcript set comes as a PSL file, which cannot be indexed. It is turned into BED12 (a line per transcript with its exons), then sorted, compressed and indexed like the others. This is the track that shows every transcript the pipeline kept.

In [4]:
for f in sorted(glob.glob(f"{out_dir}/*.gz")):
    print(check_index(f, "chr22"), "records on chr22 in", os.path.basename(f))

132 records on chr22 in SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.bed.gz
1619 records on chr22 in SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.gtf.gz
305 records on chr22 in SERVES_LRP2_test.transcriptome.filtered.bed.gz


**Explanation.** A check that each index works: asking for chr22 should return records. Change `chr22` to a chromosome in your data.

In [5]:
if bam_dir:
    for bam in sorted(glob.glob(f"{bam_dir}/*.bam")):
        print(prepare_bam(bam, out_dir))
else:
    print("bam_dir is not set: no alignments to prepare")

bam_dir is not set: no alignments to prepare


**Explanation.** Each alignment BAM is sorted by position and indexed (a `.bai` file), into the `igv/` folder. The test results contain no BAM files, so this cell does nothing until `bam_dir` points at a folder of them. In IGV use File, Load from File and pick the `.gz` and `.bam` files; IGV finds the index next to each.